In [0]:
from pyspark.sql.functions import col, when, median, mode

# 1. Cargar datos desde la capa Bronze
df_bronze = spark.read.table("workspace.default.framingham_raw")

# 2. Limpieza de Duplicados
# En salud, registros idénticos suelen ser errores de ingesta
df_clean = df_bronze.dropDuplicates(["male", "age", "education", "totChol", "sysBP", "diaBP"])

# 3. Manejo de Valores Nulos (Imputación)
# Para un modelo predictivo, no queremos perder filas, pero tampoco introducir sesgo.
# Estrategia: Mediana para continuas, Moda para categóricas.

# Calculamos valores para imputar
stats = df_clean.select(
    median("glucose").alias("med_glucose"),
    median("totChol").alias("med_totChol"),
    median("BMI").alias("med_BMI"),
    median("heartRate").alias("med_heartRate")
).collect()[0]

df_silver = df_clean.fillna({
    "glucose": stats["med_glucose"],
    "totChol": stats["med_totChol"],
    "BMI": stats["med_BMI"],
    "heartRate": stats["med_heartRate"],
    "education": 1.0, # Asumimos nivel base si falta
    "BPMeds": 0.0,    # Asumimos que no toma si no está registrado
    "cigsPerDay": 0.0 # Asumimos no fumador si falta
})

# 4. Validaciones de Calidad (Data Quality Checks)
# Filtramos registros que no tienen sentido físico (outliers extremos o errores)
df_silver = df_silver.filter(
    (col("age") > 0) & (col("age") < 120) &
    (col("sysBP") > col("diaBP")) # La sistólica siempre es mayor a la diastólica
)

# 5. Escribir a la capa Silver (Delta Lake)
# Usamos 'mergeSchema' por si en el futuro agregamos validaciones
(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.framingham_clean"))

print("Capa Silver creada exitosamente.")